In [ ]:
import hglm
import numpy as np

n_jobs = -1
n_repeat = 100
radius = 4
effect_perc = .2
hotel_tr_all = np.logspace(np.log10(.01), np.log10(5), 15)

# load human connectome project data
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp_hcp = hglm.experiment.ExperimentImageOnly.from_search(
    folder=folder,
    sbj_regex=r'[\d]{6}',
    img_glob_dict={'FA': '*_FA.nii.gz',
                   'MD': '*_MD.nii.gz'})
exp = exp_hcp.sample_x(a=2, seed=0, add_bias=True)

# # wgn
# exp = hglm.experiment.Experiment.from_gauss(seed=0,
#                                             shape=(5, 5, 5),
#                                             a=2,
#                                             b=2,
#                                             num_img=100)

# Evaluating HGLM's segmentation objective

Does HGLM's segmentation objective (removing per-voxel residuals first) improve upon Ward's?  By how much?

Approach:
- sample `n` effect extents
- for each, impose effect at multiple `hotel_tr`
- segment according to HGLM & Ward
- report the max F1 score across segmentation for each

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import pandas as pd

def build_exp_effect(seed, hotel_tr, exp, effect_perc, radius=None):
    # trim experiment
    if radius is not None:
        extenter = hglm.effect.ExtenterSphere(radius=radius)
        mask = extenter(mask_idx=exp.mask_idx, seed=seed, contiguous=True)
        exp = exp.apply_mask(mask)

    # sample extent
    n = exp.y.shape[2] * effect_perc
    extenter = hglm.effect.ExtenterMinVar(n=n)
    mask_target = extenter(y=exp.y,
                           mask_idx=exp.mask_idx,
                           seed=seed)

    # impose effect
    exp, effect = exp.impose_effect(mask=mask_target,
                                    seed=seed,
                                    hotel_tr=hotel_tr)

    return exp, effect

In [ ]:
def run_segment_test(**kwargs):
    # generate experiment
    exp, effect = build_exp_effect(**kwargs)
    
    # cluster (hglm & ward)
    d = dict()
    for mode in ('ward-full', 'ward-proj'):
        children = hglm.experiment.AnalysisHGLM.cluster(exp, mode)
        f1, sens, spec = hglm.graph.get_f1_sens_spec(mask=effect.mask,
                               mask_idx=exp.mask_idx,
                               children=children)
        
        # record scores for maxf1 region
        idx = f1.argmax()
        d[mode] = dict(f1=f1[idx], 
                            sens=sens[idx],
                            spec=spec[idx])
        
    return dict(seed=effect.seed, 
                hotel_tr=effect.hotel_tr,
                exp_size=exp.y.shape[2], **d)

In [ ]:
param_grid = list(product(range(int(n_repeat)), hotel_tr_all))

results = Parallel(n_jobs=n_jobs)(
    delayed(run_segment_test)(seed=seed, 
                             hotel_tr=hotel_tr, 
                             exp=exp, 
                             effect_perc=effect_perc, 
                             radius=radius)
    for seed, hotel_tr in tqdm(param_grid, desc='experiment (per seed-stat)')
)

# aggregate
rows = []
for r in results:
    for mode in ('ward-full', 'ward-proj'):
        rows.append({
            'seed': r['seed'],
            'hotel_tr': r['hotel_tr'],
            'exp_size': r['exp_size'],
            'mode': mode,
            **r[mode],  # f1, sens, spec
        })
df = pd.DataFrame(rows)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

metrics = {'f1': 'F1', 'sens': 'Sensitivity', 'spec': 'Specificity'}
mode_colors = {'ward-full': 'green', 'ward-proj': 'blue'}

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex='col')
for col_idx, metric in enumerate(metrics.keys()):
    ax_top = axes[0, col_idx]
    ax_bot = axes[1, col_idx]

    # --- Top row: per seed-mode lines + bold mean line per mode ---
    for mode, color in mode_colors.items():
        sub = df[df['mode'] == mode].copy()

        for seed, g in sub.groupby('seed'):
            g = g.sort_values('hotel_tr')
            ax_top.plot(g['hotel_tr'].values, g[metric].values, lw=.3, alpha=0.4, color=color)

        mean_curve = (
            sub.groupby('hotel_tr', as_index=False)[metric]
               .mean()
               .sort_values('hotel_tr')
        )
        ax_top.plot(mean_curve['hotel_tr'].values, mean_curve[metric].values,
                    lw=5, color=color, label=mode.replace('ward-', ''))

    if col_idx == 0:
        ax_top.legend(title='mode', frameon=False)
    ax_top.set_title(metrics[metric])
    ax_top.grid(True, linewidth=1.5)

    # --- Bottom row: differences (proj - full), black ---
    pivot = (
        df.pivot_table(index=['seed', 'hotel_tr'], columns='mode', values=metric)
          .reset_index()
          .sort_values(['seed', 'hotel_tr'])
    )
    pivot = pivot.dropna(subset=['ward-full', 'ward-proj'])
    pivot['diff'] = pivot['ward-proj'] - pivot['ward-full']

    for seed, g in pivot.groupby('seed'):
        ax_bot.plot(g['hotel_tr'].values, g['diff'].values, lw=.3, alpha=0.4, color='black')

    diff_mean = (
        pivot.groupby('hotel_tr', as_index=False)['diff']
             .mean()
             .sort_values('hotel_tr')
    )
    ax_bot.plot(diff_mean['hotel_tr'].values, diff_mean['diff'].values, lw=5, color='black')

    ax_bot.axhline(0, lw=1, color='black', alpha=0.3)
    ax_bot.set_xlabel('hotel_tr')
    ax_bot.grid(True, linewidth=1.5)

# --- Log scale for all subplots ---
for ax_row in axes:
    for ax in ax_row:
        ax.set_xscale('log')

# Labels for rows
axes[0, 0].set_ylabel('score')
axes[1, 0].set_ylabel('proj - full')

plt.tight_layout()
plt.show()


In [ ]:
metrics = ['f1', 'sens', 'spec']

# pivot to compare full vs proj for each seed-hotel_tr pair
df_wide = df.pivot_table(index=['seed', 'hotel_tr'],
                         columns='mode',
                         values=metrics).reset_index()

for metric in metrics:
    diff = df_wide[(metric, 'ward-proj')] - df_wide[(metric, 'ward-full')]
    p_improve = (diff > 0).mean()
    p_equal   = (diff == 0).mean()
    p_worse   = 1 - p_equal - p_improve

    print(f'=== {metric.upper()} ===')
    print(f'Proj > Full in {p_improve:.3f} of cases')
    print(f'Proj = Full in {p_equal:.3f} of cases')
    print(f'Proj < Full in {p_worse:.3f} of cases\n')


# Evaluating HGLM's tailoring objective

Does HGLM's tailoring process identify the max F1 region?

Approach:
- sample `n` effect extents
- for each, impose effect at multiple `f_ratio`
- segment according to HGLM
- choose the max F1 score, declare it and all of its ancestors & descendents as significant
- run the LL tailoring process, compare the f1 score of its output to the f1 score of the target region

todo: add sens & spec of "best" region
todo: add total sens & spec

In [ ]:
def run_tailor_test(**kwargs):
    # generate experiment
    exp, effect = build_exp_effect(**kwargs)
    
    # segment
    children = hglm.experiment.AnalysisHGLM.cluster(exp, mode='ward-proj')
    f1, sens, spec = hglm.graph.get_f1_sens_spec(mask=effect.mask,
                           mask_idx=exp.mask_idx,
                           children=children)
    
    # "hypothesis testing": declare target region & all ancestors/descendents as significant
    reg_target = f1.argmax()
    num_leaf = exp.y.shape[2]
    sig_reg_list = list(hglm.graph.iter_topo(children=children, 
                                             num_leaf=num_leaf, 
                                             node_start=reg_target))
    assert reg_target in sig_reg_list
    node = reg_target
    parent = hglm.graph.get_parent(children, num_leaf)
    while parent[node] != -1:
        p = parent[node]
        sig_reg_list.append(p)
        node = p
        
    # tailor the outputs
    reg_out_list = hglm.experiment.AnalysisHGLM.tailor(sig_reg_list=sig_reg_list,
                                                       children=children,
                                                       exp=exp,
                                                       alpha_tailor=.05,
                                                       n_perm=100)
    # get "best" single region out
    f1[reg_out_list].argmax()
        
    return dict(seed=effect.seed, 
                hotel_tr=effect.hotel_tr, 
                f1_max=f1.max(), 
                f1_max_out=f1[reg_out_list].max())

In [ ]:
param_grid = list(product(range(int(n_repeat)), hotel_tr_all))

results = Parallel(n_jobs=n_jobs)(
    delayed(run_tailor_test)(seed=seed, 
                             hotel_tr=hotel_tr, 
                             exp=exp, 
                             effect_perc=effect_perc, 
                             radius=radius)
    for seed, hotel_tr in tqdm(param_grid, desc='experiment (per seed-stat)')
)


df = pd.DataFrame(results)
df['f1_max_ratio'] = df['f1_max_out'] / df['f1_max']

In [ ]:
# --- Feature Parameters ---
x_feature = 'hotel_tr'
y_feature = 'f1_max_ratio'
group_feature = 'seed'

# --- Plotting ---
plt.figure(figsize=(10, 6))

# Plot each seed line (black, no marker, no legend)
for _, group_df in df.groupby(group_feature):
    group_df = group_df.sort_values(by=x_feature)
    plt.plot(group_df[x_feature], group_df[y_feature],
             color='grey', linewidth=1)

# Plot bold average line (black, with legend)
avg_df = df.groupby(x_feature, as_index=False)[y_feature].mean()
avg_df = avg_df.sort_values(by=x_feature)

plt.plot(avg_df[x_feature], avg_df[y_feature],
         color='black', linewidth=5, label='Average')

# Axis formatting
plt.xscale('log')
plt.xlabel(x_feature)
plt.ylabel(y_feature)
plt.title(f'{y_feature} vs {x_feature} (Averaged Across Seeds)')
plt.legend(title='', loc='best')
plt.tight_layout()
plt.show()
